In [ ]:
#Run imports for libraries to be used
import pandas as pd
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import zscore
from statistics import linear_regression
from sklearn.preprocessing import PowerTransformer
from imblearn.over_sampling import RandomOverSampler
import re
from sklearn.model_selection import cross_validate
from sklearn.impute import SimpleImputer
import sklearn


#Set up display options for pandas return functions
pd.options.display.max_rows = 100
pd.options.display.max_columns = 110
sklearn.set_config(transform_output="pandas")

In [ ]:
#load the schema
with open('ingest_schema.json') as f:
    schema = json.load(f)

#Use the schema and other params to ingest the csv
df_all = pd.read_csv(
    filepath_or_buffer ="CW_data.csv",
    encoding='ANSI',
    dtype = schema,
    true_values = ["positive", "detected"],
    false_values = ["negative", "not_detected"],
    on_bad_lines = "warn"
    )
#clean column labels for invisible space characters, create function for use on unseen data.
def clean_cols(df):
    # Remove occurrences of \xa0
    df.columns = [re.sub(r'\s+', ' ', col).strip() for col in df.columns]
    return df
df_all = clean_cols(df_all)
#drop bad cols & Rows (ID, NaN cols)
df_all.drop('Patient ID', axis = 1, inplace = True)
df_single_values =  df_all.loc[:, df_all.nunique(dropna = False) == 1]
df_all = df_all.drop(labels = df_single_values.columns, axis = 1)
ignored_cols = ["Patient age quantile", "SARS-Cov-2 exam result"]
all_feats_cols = df_all.columns.difference(ignored_cols)
df_all = df_all.dropna(how = "all", subset = all_feats_cols)

#instantiate static items
transformer = PowerTransformer(method = 'yeo-johnson')
transformer.set_output(transform="pandas")
regression = LogisticRegression(random_state = 1)
rfclf = RandomForestClassifier(random_state = 1)
knn = KNeighborsClassifier()
skf = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 1)
imputer = SimpleImputer(strategy = 'median')

In [ ]:
def prepare_data(nan_threshold, over_sample_method, pca_components):
    # Threshold values for NaN Cols
    missing_per_column = df_all.isnull().sum(axis=0)
    threshold = missing_per_column < nan_threshold
    df_thresholded = df_all.loc[:,threshold]
    # Split Classifiers
    df_classifiers = df_thresholded['SARS-Cov-2 exam result']
    df_thresholded.drop('SARS-Cov-2 exam result', axis = 1).copy()
    #splitting test and train sets
    predictor_train, predictor_test, classifier_train, classifier_test = train_test_split(df_thresholded,df_classifiers, test_size = 0.2, random_state = 1)
    # imputing
    predictor_train = imputer.fit_transform(predictor_train)
    predictor_test = imputer.transform(predictor_test)
    # Skewing
    cols_for_transform = df_thresholded.select_dtypes(exclude = ['boolean']).columns
    transformed_data = transformer.fit_transform(predictor_train[cols_for_transform].astype(float))
    transformed_data_test = transformer.transform(predictor_test[cols_for_transform].astype(float))
    predictor_train[cols_for_transform] = transformed_data
    predictor_test[cols_for_transform] = transformed_data_test
    # PCA
    pca = PCA(n_components = pca_components)
    predictor_train = pca.fit_transform(predictor_train)
    predictor_test = pca.transform(predictor_test)
    # Oversampling
    random_oversample = RandomOverSampler(random_state = 1, sampling_strategy = over_sample_method)
    predictor_train, classifier_train = random_oversample.fit_resample(predictor_train, classifier_train)
    return predictor_train, classifier_train, predictor_test, classifier_test

def run_models(predictor_train, classifier_train, predictor_test):
    regression.fit(predictor_train, classifier_train)
    predictions_regression = regression.predict(predictor_test)
    rfclf.fit(predictor_train, classifier_train)
    predictions_rfclf = rfclf.predict(predictor_test)
    knn.fit(predictor_train, classifier_train)
    predictions_knn = knn.predict(predictor_test)
    return predictions_regression, predictions_rfclf,predictions_knn, regression, rfclf, knn

def cross_val(model, X, y):
    results_cv = cross_validate(model, X, y, cv = skf, scoring = ['f1'], return_train_score = False)
    model_name = model.__class__.__name__
    score = f'F1 score Mean for {model_name} model was: {results_cv['test_f1'].mean()}'
    return score





In [ ]:
# prepare data and pass the returned splits as vars
pred_train, class_train, pred_test, class_test = prepare_data(
    nan_threshold = 1100,
    over_sample_method = .5,
    pca_components = 5 )

#Run models
predictions_regression, predictions_rfclf,predictions_knn, regression_model, rf_model, knn_model = run_models(pred_train, class_train, pred_test)

print(f'Scores are:\n{cross_val(regression_model, pred_test, class_test)} \n{cross_val(rf_model, pred_test, class_test)}\n{cross_val(knn_model, pred_test, class_test)}')
